# Cleaning the Online Retail II Dataset

This notebook takes the raw invoice-level file and turns it into an analysis-ready table.
The goal is not just to remove bad rows, it's to write down every decision as I make it,
so the assumptions log actually means something later when I'm explaining choices in an interview.

Each section below does three things: shows what the problem looks like in the raw data,
makes a call on how to handle it, and logs how many rows were affected.

The dataset is UK-based online gift retail, invoice line items from December 2009 to December 2011.


## Step 0: Load the raw file

Nothing fancy here, just reading the CSV and confirming it matches what the data source describes.
Customer ID is read in as a string on purpose. It's an identifier, not a number, and pandas will
otherwise turn it into a float and add a trailing .0 to everything.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../datasets/raw/online_retail_II.csv', encoding='latin1', dtype={'Customer ID': str})
print("Rows loaded:", len(df))
print("Columns:", list(df.columns))
print("Date range:", df['InvoiceDate'].min(), "to", df['InvoiceDate'].max())
print("Unique customers:", df['Customer ID'].nunique())
print("Unique invoices:", df['Invoice'].nunique())

start_rows = len(df)
log_rows = []  # will hold (decision, rows_affected, rows_remaining) for the assumptions log


Rows loaded: 1067371
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Unique customers: 5942
Unique invoices: 53628

## Step 1: Check what's actually missing

Before touching anything, it helps to just look at where the nulls are. `Customer ID` is the
one that matters most since a lot of the later analysis (RFM, cohort, churn) only makes sense
per customer.


In [2]:
print(df.isnull().sum())


Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

## Step 2: Non-product rows

Not every row in this file is a product being sold. Some `StockCode` values are things like
postage, manual entries, bank charges, or a discount line. If these get left in, revenue totals
and product-level analysis both end up wrong, since they're not really "products" a customer bought.

Decision: pull these out into their own step and keep the rest as `df_products`. I'm not deleting
them from existence, just separating them from the product-level table so they don't distort
things like average order value or top-selling products.


In [3]:
non_product_codes = ['POST', 'DOT', 'M', 'm', 'D', 'BANK CHARGES', 'C2',
                      'ADJUST', 'AMAZONFEE', 'CRUK', 'PADS', 'GIFT', 'S', 'B',
                      'DCGSSGIRL', 'DCGSSBOY']
non_product_mask = df['StockCode'].isin(non_product_codes)
print("Rows flagged as non-product (postage, manual, discount, fees, etc):", non_product_mask.sum())
print(df.loc[non_product_mask, 'StockCode'].value_counts())

df_products = df.loc[~non_product_mask].copy()
removed = non_product_mask.sum()
log_rows.append(("Removed non-product StockCodes (POST, DOT, M, D, BANK CHARGES, etc)",
                  removed, len(df_products)))
print("\nRows remaining after removing non-product codes:", len(df_products))


Rows flagged as non-product (postage, manual, discount, fees, etc): 5859
StockCode
POST            2122
DOT             1446
M               1421
C2               282
D                177
S                104
BANK CHARGES     102
ADJUST            67
AMAZONFEE         43
DCGSSGIRL         25
DCGSSBOY          23
PADS              19
CRUK              16
B                  6
m                  5
GIFT               1
Name: count, dtype: int64

Rows remaining after removing non-product codes: 1061512

## Step 3: Cancellations

Any invoice starting with the letter `C` is a cancellation, and it's paired with a negative
quantity. These are real, legitimate returns, so they're not junk data, but they don't belong
mixed in with regular purchases when I'm counting how many times a customer actually bought
something.

Decision: pull cancellations into their own table (`df_cancellations`) rather than throwing them
away. If a later question needs return behavior specifically, it's there. For the main analysis
they're excluded.


In [4]:
cancel_mask = df_products['Invoice'].str.startswith('C', na=False)
print("Cancellation rows (Invoice starts with C):", cancel_mask.sum())
df_cancellations = df_products.loc[cancel_mask].copy()
df_no_cancel = df_products.loc[~cancel_mask].copy()
log_rows.append(("Removed cancellation invoices (Invoice starts with 'C')",
                  cancel_mask.sum(), len(df_no_cancel)))
print("Rows remaining after removing cancellations:", len(df_no_cancel))
print("\nNote: cancellations are kept in a separate dataframe (df_cancellations) rather than")
print("thrown away, in case a later analysis wants to look at return behavior specifically.")


Cancellation rows (Invoice starts with C): 18290
Rows remaining after removing cancellations: 1043222

Note: cancellations are kept in a separate dataframe (df_cancellations) rather than
thrown away, in case a later analysis wants to look at return behavior specifically.

## Step 4: Negative quantity that isn't a cancellation

This one is not mentioned as directly in most write-ups of this dataset, but it's real. There are
rows with negative quantity where the invoice does not start with `C`. Looking at the
`Description` field on these rows tells the story: values like "damaged", "lost", "missing",
"thrown away", "smashed". These are internal stock write-offs, not customer cancellations.

Decision: exclude these from the clean dataset. They're not a sale and they're not a customer
return either, they're inventory shrinkage. Worth a mention in the executive summary as a
separate operational note, but they shouldn't affect churn, RFM, or revenue numbers.


In [5]:
phantom_mask = df_no_cancel['Quantity'] < 0
print("Negative quantity rows that are NOT cancellations (stock adjustments/write-offs):",
      phantom_mask.sum())
print(df_no_cancel.loc[phantom_mask, 'Description'].value_counts().head(15))

df_clean_step = df_no_cancel.loc[~phantom_mask].copy()
log_rows.append(("Removed negative-quantity rows without a cancellation invoice (stock write-offs, damages, lost stock)",
                  phantom_mask.sum(), len(df_clean_step)))
print("\nRows remaining:", len(df_clean_step))


Negative quantity rows that are NOT cancellations (stock adjustments/write-offs): 3454
Description
check                     123
damages                    84
?                          83
damaged                    78
missing                    27
sold as set on dotcom      20
Damaged                    17
smashed                     9
thrown away                 9
Unsaleable, destroyed.      9
dotcom                      8
damages?                    7
??                          7
crushed                     6
given away                  6
Name: count, dtype: int64

Rows remaining: 1039768

## Step 5: Zero or negative price

A handful of rows have a price of zero or less. Some of these are free samples, some look like
data entry mistakes. Either way they add nothing to revenue and would just create noise in
average order value and RFM's monetary score.

Decision: drop them from the clean dataset.


In [6]:
bad_price_mask = df_clean_step['Price'] <= 0
print("Rows with Price <= 0:", bad_price_mask.sum())
df_clean_step2 = df_clean_step.loc[~bad_price_mask].copy()
log_rows.append(("Removed rows with Price <= 0 (free samples, price entry errors)",
                  bad_price_mask.sum(), len(df_clean_step2)))
print("Rows remaining:", len(df_clean_step2))


Rows with Price <= 0: 2718
Rows remaining: 1037050

## Step 6: Exact duplicates

Some rows are byte-for-byte identical across every column. This is different from a customer
legitimately ordering two of the same item on separate lines, an exact duplicate means the same
row got logged twice.

Decision: drop exact duplicates, keep everything else as is.


In [7]:
dup_mask = df_clean_step2.duplicated()
print("Exact duplicate rows found:", dup_mask.sum())
df_deduped = df_clean_step2.drop_duplicates().copy()
log_rows.append(("Removed exact duplicate rows",
                  dup_mask.sum(), len(df_deduped)))
print("Rows remaining:", len(df_deduped))


Exact duplicate rows found: 33664
Rows remaining: 1003386

## Step 7: Add a couple of columns that make later work easier

`InvoiceDate` comes in as a plain string, so it gets converted to an actual datetime. `LineTotal`
is just quantity times price, added now so every later notebook can use it directly instead of
recomputing it.


In [8]:
df_deduped['InvoiceDate'] = pd.to_datetime(df_deduped['InvoiceDate'])
df_deduped['LineTotal'] = df_deduped['Quantity'] * df_deduped['Price']
print("Added InvoiceDate as datetime and LineTotal (Quantity * Price)")
print(df_deduped[['Invoice', 'StockCode', 'Quantity', 'Price', 'LineTotal', 'InvoiceDate']].head(5))


Added InvoiceDate as datetime and LineTotal (Quantity * Price)
  Invoice StockCode  Quantity  Price  LineTotal         InvoiceDate
0  489434     85048        12   6.95       83.4 2009-12-01 07:45:00
1  489434    79323P        12   6.75       81.0 2009-12-01 07:45:00
2  489434    79323W        12   6.75       81.0 2009-12-01 07:45:00
3  489434     22041        48   2.10      100.8 2009-12-01 07:45:00
4  489434     21232        24   1.25       30.0 2009-12-01 07:45:00

## Step 8: Splitting out guest orders (no Customer ID)

About a fifth of the remaining rows have no `Customer ID`. These are real orders, they're just
not tied to an identifiable customer, so there's no way to include them in anything customer-level
like RFM segments, cohort retention, or churn.

Decision: split these off into their own table (`df_guest_orders`). They stay usable for
revenue and product-level questions ("what sold well in March"), but they're excluded from
anything that needs to track an individual customer over time.


## Step 7b: Fix Customer ID formatting

`Customer ID` came in from the source file as a float, so every value has a trailing `.0`
(`13085.0` instead of `13085`). It's read as text here to avoid pandas dropping the decimal
silently, but the `.0` itself needs a manual fix. Left alone, this becomes a real problem
later. It is the join key for the customer dimension in Power BI, and a float-formatted key
either breaks the join or renders as `13085.00` in every slicer and drill-through.

Fixing it now, before the guest-order split, so it's clean in every table downstream instead
of needing to be re-fixed four separate times.


In [ ]:
def clean_customer_id(x):
    if pd.isna(x):
        return x
    return str(int(float(x)))

df_deduped['Customer ID'] = df_deduped['Customer ID'].apply(clean_customer_id)
print("Sample Customer ID values after fix:", df_deduped['Customer ID'].dropna().unique()[:5].tolist())


Sample Customer ID values after fix: ['13085', '13078', '15362', '18102', '12682']


In [9]:
null_cust_mask = df_deduped['Customer ID'].isna()
print("Rows with no Customer ID (guest/unlinked orders):", null_cust_mask.sum(),
      f"({null_cust_mask.mean()*100:.1f}% of remaining rows)")

df_customer_level = df_deduped.loc[~null_cust_mask].copy()
df_guest_orders = df_deduped.loc[null_cust_mask].copy()

log_rows.append(("Set aside rows with no Customer ID into a separate guest-orders table (kept for revenue/product-level work, excluded from customer-level work)",
                  null_cust_mask.sum(), len(df_customer_level)))

print("\nRows kept for customer-level analysis (RFM, cohort, churn):", len(df_customer_level))
print("Rows kept separately for product/revenue-level analysis (guest orders):", len(df_guest_orders))


Rows with no Customer ID (guest/unlinked orders): 226794 (22.6% of remaining rows)

Rows kept for customer-level analysis (RFM, cohort, churn): 776592
Rows kept separately for product/revenue-level analysis (guest orders): 226794

## Step 9: Where things ended up

A quick summary of everything that happened above, plus the full assumptions log printed out
in one place.


In [10]:
print(f"Started with:            {start_rows:,} rows")
print(f"Ended with (all clean):  {len(df_deduped):,} rows")
print(f"  -> customer-level:     {len(df_customer_level):,} rows")
print(f"  -> guest-orders only:  {len(df_guest_orders):,} rows")
print(f"Total removed:           {start_rows - len(df_deduped):,} rows "
      f"({(start_rows - len(df_deduped)) / start_rows * 100:.1f}%)")

print("\nAssumptions log:")
for step, affected, remaining in log_rows:
    print(f"- {step}")
    print(f"  Rows affected: {affected:,} | Rows remaining after this step: {remaining:,}")


Started with:            1,067,371 rows
Ended with (all clean):  1,003,386 rows
  -> customer-level:     776,592 rows
  -> guest-orders only:  226,794 rows
Total removed:           63,985 rows (6.0%)

Assumptions log:
- Removed non-product StockCodes (POST, DOT, M, D, BANK CHARGES, etc)
  Rows affected: 5,859 | Rows remaining after this step: 1,061,512
- Removed cancellation invoices (Invoice starts with 'C')
  Rows affected: 18,290 | Rows remaining after this step: 1,043,222
- Removed negative-quantity rows without a cancellation invoice (stock write-offs, damages, lost stock)
  Rows affected: 3,454 | Rows remaining after this step: 1,039,768
- Removed rows with Price <= 0 (free samples, price entry errors)
  Rows affected: 2,718 | Rows remaining after this step: 1,037,050
- Removed exact duplicate rows
  Rows affected: 33,664 | Rows remaining after this step: 1,003,386
- Set aside rows with no Customer ID into a separate guest-orders table (kept for revenue/product-level work, exclud

## Step 10: Save everything

Four files come out of this notebook:

- the full cleaned table, cancellations and non-product rows removed but guests still included
- the customer-level table, used for RFM, cohort, and churn in the next notebook
- the guest-orders table, kept aside for product and revenue questions only
- the cancellations table, kept in case a return-behavior question comes up later


In [11]:
df_deduped.to_csv('../datasets/cleaned/orders_clean_full.csv', index=False)
df_customer_level.to_csv('../datasets/cleaned/orders_clean_customer_level.csv', index=False)
df_guest_orders.to_csv('../datasets/cleaned/orders_guest_only.csv', index=False)
df_cancellations.to_csv('../datasets/cleaned/orders_cancellations.csv', index=False)
print("Saved:")
print(" - data/cleaned/orders_clean_full.csv")
print(" - data/cleaned/orders_clean_customer_level.csv")
print(" - data/cleaned/orders_guest_only.csv")
print(" - data/cleaned/orders_cancellations.csv")


Saved:
 - data/cleaned/orders_clean_full.csv
 - data/cleaned/orders_clean_customer_level.csv
 - data/cleaned/orders_guest_only.csv
 - data/cleaned/orders_cancellations.csv

Wrote docs/assumptions_log.md

## What's next

`orders_clean_customer_level.csv` is what feeds into the SQL business queries and then the
RFM, churn, and cohort work in the next notebook. The full assumptions log from this notebook
also gets written out to `docs/assumptions_log.md` so it's easy to point to later without
digging back through code.

One thing worth flagging for the executive summary: there's a single very large line item
(a discount/manual entry over 25,000 in value) and one line item worth over 168,000 in revenue.
Both were kept in the clean data since they're legitimate wholesale-sized orders, not errors,
but they're big enough that they're worth a sentence in the write-up rather than letting them
silently skew the averages.
